# 00 — Environment Setup

Install only missing dependencies, clone the repo, verify GPU, and prepare full BIRD dev. Google Drive is optional and not required on Kaggle.


In [ ]:
import importlib.util
import os
import subprocess
import sys

# Kaggle usually ships torch. Install only packages that are missing.
# vLLM is intentionally optional here: it is large and can conflict with the
# CUDA/PyTorch stack already preinstalled by Kaggle. Enable it explicitly with
# INSTALL_VLLM=1 when running inference notebooks (02/04).
PACKAGE_IMPORTS = {
    "datasets": "datasets",
    "huggingface_hub": "huggingface_hub",
    "peft": "peft",
    "trl": "trl",
    "unsloth": "unsloth",
    "scikit-learn": "sklearn",
    "xgboost": "xgboost",
    "scipy": "scipy",
    "seaborn": "seaborn",
    "sqlglot": "sqlglot",
    "pytest": "pytest",
}

missing = [
    package
    for package, module in PACKAGE_IMPORTS.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    # --no-cache-dir avoids filling the limited Kaggle disk with pip wheels.
    # Kaggle must have Notebook Settings -> Internet enabled for this step.
    install_names = ["unsloth[kaggle]" if p == "unsloth" else p for p in missing]
    print(f"Installing missing packages: {install_names}")
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *install_names]
        )
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Package installation failed. On Kaggle, enable "
            "Notebook Settings -> Internet, restart the session, and rerun "
            "this cell. The error is usually DNS/network access, not Python "
            "code. If Internet cannot be enabled, upload/install the required "
            "wheels as a Kaggle Dataset instead."
        ) from exc

# Re-check after pip install. This avoids reporting a stale pre-install state.
missing_after_install = [
    package
    for package, module in PACKAGE_IMPORTS.items()
    if importlib.util.find_spec(module) is None
]
if missing_after_install:
    raise RuntimeError(
        "These packages are still unavailable after installation: "
        + ", ".join(missing_after_install)
    )

install_vllm = os.environ.get("INSTALL_VLLM", "0").strip().lower() in {
    "1", "true", "yes"
}
if install_vllm and importlib.util.find_spec("vllm") is None:
    print("INSTALL_VLLM=1: installing vLLM (this can take several minutes)...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "vllm"]
    )

vllm_available = importlib.util.find_spec("vllm") is not None
print("Packages ready:", ", ".join(PACKAGE_IMPORTS))
if vllm_available:
    print("vLLM: available")
else:
    print(
        "vLLM: not installed (this is not an error for 00_setup; "
        "set INSTALL_VLLM=1 before rerunning this cell for 02/04)"
    )


In [ ]:
import torch

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'GPU {i}: {props.name}, {props.total_memory / 1024**3:.1f} GB VRAM')
        print(f'  Compute capability: {props.major}.{props.minor}')
    print(f'PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}')
else:
    print('WARNING: No GPU detected. SFT/vLLM cells require a Kaggle GPU session.')


In [ ]:
# No Google Drive mount is required on Kaggle. Drive is intentionally optional:
# use a Kaggle Dataset or Hugging Face Hub for artifacts between sessions.
from pathlib import Path

print('Kaggle working directory:', Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd())


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# This cell is self-contained so 00_setup also works from a blank Kaggle session.
REPO_URL = os.environ.get('SLM_PRJ_REPO_URL', 'https://github.com/minhadcaz/SLMSQL_project.git')

def _is_repo(path):
    return (path / 'pyproject.toml').is_file() and (path / 'benchmarks').is_dir()

candidates = [Path.cwd(), *Path.cwd().resolve().parents,
              Path('/kaggle/working/SLM-PRJ'), Path('/kaggle/working/SLMSQL_project')]
REPO_DIR = next((p for p in candidates if _is_repo(p)), None)
if REPO_DIR is None:
    REPO_DIR = Path('/kaggle/working/SLM-PRJ')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR))
from scripts.kaggle_utils import prepare_repo_import
REPO_DIR = prepare_repo_import(REPO_DIR)
print(f'Repository: {REPO_DIR}')

In [ ]:
import os
from scripts.kaggle_utils import (
    prepare_bird_dev,
    resolve_data_dir,
    resolve_synsql_raw,
    validate_bird_dir,
)

DATA_DIR = resolve_data_dir(REPO_DIR)
BIRD_DIR = DATA_DIR / 'bird'
SPIDER_DIR = DATA_DIR / 'spider'
SYNSQL_DIR = DATA_DIR / 'synsql'
SYNSQL_RAW = resolve_synsql_raw(REPO_DIR)

# Full BIRD dev is the primary benchmark. An attached Kaggle dataset is used
# first; otherwise the official development archive is downloaded.
bird_info = prepare_bird_dev(
    DATA_DIR,
    force_refresh=os.environ.get('FORCE_REFRESH_DATA', '0') == '1',
)
print(f'BIRD dev ready: {bird_info["detail"]}')
print(f'BIRD path: {BIRD_DIR}')

# Spider and synthetic training data are optional. Do not force a large SynSQL
# download: attach a curated file or set SYNSQL_RAW when SFT is needed.
print(f'Spider: {"ready" if (SPIDER_DIR / "dev.json").is_file() else "not attached (optional)"}')
print(f'SynSQL/Syn Think: {SYNSQL_RAW if SYNSQL_RAW else "not supplied (SFT notebook will explain how to attach it)"}')


In [ ]:
from benchmarks.bird_loader import load_bird_dev, get_db_path
from benchmarks.eval_ex import execution_match, batch_evaluate
from training.data_prep import filter_synsql, prepare_subsets
from training.prompt_template import format_sft_messages
from scripts.kaggle_utils import validate_bird_dir

ok, detail = validate_bird_dir(BIRD_DIR)
assert ok, detail
print('BIRD validation:', detail)
print('Core imports OK.')


In [ ]:
import subprocess

subprocess.run([
    sys.executable, '-m', 'pytest',
    'tests/test_eval_ex.py', 'tests/test_data_prep.py', '-q',
], check=True)
